----------------------------
### perform sentiment analysis with Sentence Transformers. 

- We will read individual text files for sentiment classification and use Keras layers for building the classifier.

------------------------------

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

from tensorflow import keras

from tensorflow.keras.datasets import imdb

from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.layers import Input, Dense, Embedding, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from sklearn.metrics import accuracy_score

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
import os
os.environ["SENTENCE_TRANSFORMERS_HOME"] = r'D:\Makesh\Working\AI\RPS\Day04\Dataset\sentence-transformers'

In [ ]:
# Load a pre-trained Sentence Transformer model
model = SentenceTransformer('bert-base-nli-mean-tokens')

In [ ]:
# Load the IMDb dataset from Keras
num_words = 10000  # Top most frequent words to consider
max_len   = 256  # Maximum sequence length

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=num_words)

x_train, y_train = x_train[:500], y_train[:500]
x_test, y_test   = x_test[:100],  y_test[:100]
x_train.shape, x_test.shape
x_train[0], y_train[0]

In [ ]:
# Pad sequences to a fixed length
x_train = pad_sequences(x_train, maxlen=max_len)
x_test  = pad_sequences(x_test, maxlen=max_len)
x_train.shape, x_test.shape


In [ ]:
# Convert the IMDb dataset into text
def convert_to_text(word_indexes):
    word_index = imdb.get_word_index()
    reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in word_indexes])

In [ ]:
%time
# takes some time, abt 5 mins for 500 samples !!
x_train_text = [convert_to_text(x) for x in x_train]
x_test_text  = [convert_to_text(x) for x in x_test]


In [ ]:
x_train_text[0], y_train[0]

In [ ]:

%time
# takes time to encode ,abt 3 mins
# Encode text using the Sentence Transformer model
x_train_embeddings = model.encode(x_train_text, convert_to_tensor=True)
x_test_embeddings  = model.encode(x_test_text, convert_to_tensor=True)

In [ ]:
# Define a Keras model for sentiment classification
input_layer  = Input(shape=(model.get_sentence_embedding_dimension(),), name='input_layer')
output_layer = Dense(1, activation='sigmoid', name='output_layer')(input_layer)

In [ ]:
model_ml = Model(inputs=input_layer, outputs=output_layer)
model_ml.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

#### visualize the ML model

In [ ]:
from tensorflow.keras.utils import plot_model

In [ ]:
#pip install pydot

In [ ]:
# Visualize the Keras model
plot_model(model_ml, to_file='sentiment_model.png', show_shapes=True, show_layer_names=True)

#### build the model

In [ ]:
%time
# Train the model
model_ml.fit(x_train_embeddings.numpy(), 
          y_train,
          epochs=50, 
          batch_size=32, 
          validation_split=0.1)


In [ ]:
# Evaluate the model on the test data
predicted_labels = model_ml.predict(x_test_embeddings.numpy())
predicted_labels = np.round(predicted_labels).flatten().astype(int)

In [ ]:
# Calculate accuracy
accuracy = accuracy_score(y_test, predicted_labels)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

Model Inference with new data

In [ ]:
word_index = imdb.get_word_index()

def encode_review(text):
    words = text.lower().split()
    encoded = []
    for w in words:
        if w in word_index:
            encoded.append(word_index[w] + 3)   # IMDB reserves 0,1,2
        else:
            encoded.append(2)  # 2 = "UNK"
    return encoded

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

maxlen = 256

def prepare_input(text):
    encoded = encode_review(text)
    padded = pad_sequences([encoded], maxlen=maxlen, padding='post')
    return padded

In [68]:
#text = "This movie was amazing and I really loved it!"
text = "feel good movie to watch"
input_data = prepare_input(text)
input_data.shape


(1, 256)

In [69]:
emb = model.encode(text)

prediction = model_ml.predict(emb.reshape(1, -1))[0][0]
print("Prediction Score:", prediction)

if prediction >= 0.5:
    print("Sentiment: Positive")
else:
    print("Sentiment: Negative")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
Prediction Score: 0.9773018
Sentiment: Positive
